In [1]:
import pandas as pd

voxceleb2_noise_dataset = pd.read_csv('/nvme2/hungdx/Lightning-hydra/data/Wild_Spoof_Dataset/voxceleb2_noise_dataset/protocol_voxceleb2.txt', sep=' ')

spoof_celeb_dataset = pd.read_csv('/nvme2/hungdx/Lightning-hydra/data/Wild_Spoof_Dataset/spoofceleb/protocol.txt', sep=' ', header=None)
spoof_celeb_dataset.columns = ['file_name', 'subset', 'label']


# Make a table to display the number of samples in each subset & label 

def create_data_distribution_table(dataset, dataset_name="Dataset"):
    """
    Create a beautiful table showing the distribution of samples across subsets and labels.
    
    Args:
        dataset (pd.DataFrame): The dataset to analyze
        dataset_name (str): Name of the dataset for display purposes
    
    Returns:
        pd.DataFrame: A formatted table with sample counts
    """
    import pandas as pd
    import numpy as np
    
    # Create cross-tabulation of subset and label
    crosstab = pd.crosstab(dataset['subset'], dataset['label'], margins=True)
    
    # Create a summary table
    summary_data = []
    
    # Add rows for each subset
    for subset in dataset['subset'].unique():
        subset_data = dataset[dataset['subset'] == subset]
        label_counts = subset_data['label'].value_counts()
        
        row = {
            'Dataset': dataset_name,
            'Subset': subset,
            'Total': len(subset_data)
        }
        
        # Add counts for each label
        for label in dataset['label'].unique():
            row[f'{label}_count'] = label_counts.get(label, 0)
            row[f'{label}_%'] = f"{(label_counts.get(label, 0) / len(subset_data) * 100):.1f}%"
        
        summary_data.append(row)
    
    # Add total row
    total_row = {
        'Dataset': dataset_name,
        'Subset': 'TOTAL',
        'Total': len(dataset)
    }
    
    for label in dataset['label'].unique():
        label_count = (dataset['label'] == label).sum()
        total_row[f'{label}_count'] = label_count
        total_row[f'{label}_%'] = f"{(label_count / len(dataset) * 100):.1f}%"
    
    summary_data.append(total_row)
    
    return pd.DataFrame(summary_data)

def display_beautiful_table(dataset, dataset_name="Dataset", show_percentages=True):
    """
    Display a beautiful, formatted table with the data distribution.
    
    Args:
        dataset (pd.DataFrame): The dataset to analyze
        dataset_name (str): Name of the dataset for display purposes
        show_percentages (bool): Whether to show percentage columns
    """
    import pandas as pd
    from IPython.display import display, HTML
    
    # Get the distribution table
    dist_table = create_data_distribution_table(dataset, dataset_name)
    
    # Style the table
    styled_table = dist_table.style.set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#4472C4'), ('color', 'black'), 
                                    ('font-weight', 'bold'), ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'center')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', 'black')]},
        {'selector': 'tr:nth-child(odd)', 'props': [('background-color', 'black')]},
        {'selector': 'tr:last-child', 'props': [('background-color', 'black'), 
                                               ('font-weight', 'bold')]}
    ])
    
    # Format numbers with thousands separators
    numeric_cols = [col for col in dist_table.columns if col.endswith('_count') or col == 'Total']
    for col in numeric_cols:
        styled_table = styled_table.format({col: '{:,}'})
    
    # Display the styled table
    print(f"\n📊 {dataset_name.upper()} DATA DISTRIBUTION")
    print("=" * 60)
    display(styled_table)
    
    # Print summary statistics
    print(f"\n📈 SUMMARY STATISTICS:")
    print(f"   • Total samples: {len(dataset):,}")
    print(f"   • Number of subsets: {dataset['subset'].nunique()}")
    print(f"   • Number of labels: {dataset['label'].nunique()}")
    print(f"   • Labels: {', '.join(dataset['label'].unique())}")
    
    # Show label distribution
    print(f"\n🏷️  LABEL DISTRIBUTION:")
    label_dist = dataset['label'].value_counts()
    for label, count in label_dist.items():
        percentage = (count / len(dataset)) * 100
        print(f"   • {label}: {count:,} ({percentage:.1f}%)")

# Display the distribution for spoof_celeb_dataset
display_beautiful_table(spoof_celeb_dataset, "SpoofCeleb Dataset")



📊 SPOOFCELEB DATASET DATA DISTRIBUTION


,Dataset,Subset,Total,bonafide_count,bonafide_%,spoof_count,spoof_%
0,SpoofCeleb Dataset,train,2540421,230948,9.1%,"2,309,473",90.9%
1,SpoofCeleb Dataset,dev,55741,7963,14.3%,"47,778",85.7%
2,SpoofCeleb Dataset,eval,91130,9113,10.0%,"82,017",90.0%
3,SpoofCeleb Dataset,TOTAL,2687292,248024,9.2%,"2,439,268",90.8%



📈 SUMMARY STATISTICS:
   • Total samples: 2,687,292
   • Number of subsets: 3
   • Number of labels: 2
   • Labels: bonafide, spoof

🏷️  LABEL DISTRIBUTION:
   • spoof: 2,439,268 (90.8%)
   • bonafide: 248,024 (9.2%)


In [2]:
display_beautiful_table(voxceleb2_noise_dataset, "voxceleb2_noise Dataset")


📊 VOXCELEB2_NOISE DATASET DATA DISTRIBUTION


,Dataset,Subset,Total,bonafide_count,bonafide_%
0,voxceleb2_noise Dataset,train,14991,"14,991",100.0%
1,voxceleb2_noise Dataset,dev,45008,"45,008",100.0%
2,voxceleb2_noise Dataset,eval,36237,"36,237",100.0%
3,voxceleb2_noise Dataset,TOTAL,96236,"96,236",100.0%



📈 SUMMARY STATISTICS:
   • Total samples: 96,236
   • Number of subsets: 3
   • Number of labels: 1
   • Labels: bonafide

🏷️  LABEL DISTRIBUTION:
   • bonafide: 96,236 (100.0%)


In [3]:
# Switch voxceleb2_noise_dataset train to development
voxceleb2_noise_dataset.loc[voxceleb2_noise_dataset['subset'] == 'train', 'subset'] = 'development'

# Switch voxceleb2_noise_dataset dev to train
voxceleb2_noise_dataset.loc[voxceleb2_noise_dataset['subset'] == 'dev', 'subset'] = 'train'

# Switch voxceleb2_noise_dataset development to dev
voxceleb2_noise_dataset.loc[voxceleb2_noise_dataset['subset'] == 'development', 'subset'] = 'dev'


display_beautiful_table(voxceleb2_noise_dataset, "voxceleb2_noise Dataset")


📊 VOXCELEB2_NOISE DATASET DATA DISTRIBUTION


,Dataset,Subset,Total,bonafide_count,bonafide_%
0,voxceleb2_noise Dataset,dev,14991,"14,991",100.0%
1,voxceleb2_noise Dataset,train,45008,"45,008",100.0%
2,voxceleb2_noise Dataset,eval,36237,"36,237",100.0%
3,voxceleb2_noise Dataset,TOTAL,96236,"96,236",100.0%



📈 SUMMARY STATISTICS:
   • Total samples: 96,236
   • Number of subsets: 3
   • Number of labels: 1
   • Labels: bonafide

🏷️  LABEL DISTRIBUTION:
   • bonafide: 96,236 (100.0%)


# Combined version

In [ ]:
# Combined version

# Update the file_name column for combined dataset
voxceleb2_noise_dataset['file_name'] = voxceleb2_noise_dataset['file_name'].apply(lambda x: f'voxceleb2_noise_dataset/{x}')
spoof_celeb_dataset['file_name'] = spoof_celeb_dataset['file_name'].apply(lambda x: f'spoofceleb/{x}')

combined_dataset = pd.concat([voxceleb2_noise_dataset, spoof_celeb_dataset], ignore_index=True)

display_beautiful_table(combined_dataset, "Combined Dataset")



📊 COMBINED DATASET DATA DISTRIBUTION


,Dataset,Subset,Total,bonafide_count,bonafide_%,spoof_count,spoof_%
0,Combined Dataset,dev,70732,22954,32.5%,"47,778",67.5%
1,Combined Dataset,train,2585429,275956,10.7%,"2,309,473",89.3%
2,Combined Dataset,eval,127367,45350,35.6%,"82,017",64.4%
3,Combined Dataset,TOTAL,2783528,344260,12.4%,"2,439,268",87.6%



📈 SUMMARY STATISTICS:
   • Total samples: 2,783,528
   • Number of subsets: 3
   • Number of labels: 2
   • Labels: bonafide, spoof

🏷️  LABEL DISTRIBUTION:
   • spoof: 2,439,268 (87.6%)
   • bonafide: 344,260 (12.4%)


In [5]:
# Write combined dataset to csv
combined_dataset.to_csv('/nvme2/hungdx/Lightning-hydra/data/Wild_Spoof_Dataset/combined_protocol.txt', index=False, sep=' ', header=False)

# Balacned version

In [27]:
voxceleb2_noise_dataset['file_name'] = voxceleb2_noise_dataset['file_name'].apply(lambda x: f'voxceleb2_noise_dataset/{x}')
spoof_celeb_dataset['file_name'] = spoof_celeb_dataset['file_name'].apply(lambda x: f'spoofceleb/{x}')

# Based on number of samples in train subset from voxceleb2_noise_dataset, please select randomly bonafide samples portion of samples from spoof_celeb_dataset, also randomly select spoof samples from spoof_celeb_dataset (subset is train) to make sure the number of spoof samples is the same as the number of bonafide samples

def create_balanced_train_dataset(voxceleb2_dataset, spoof_celeb_dataset, random_state=42):
    """
    Create a balanced training dataset with specific sampling requirements:
    1. 45,008 bonafide samples from SpoofCeleb train subset
    2. 45,008 bonafide samples from VoxCeleb2 train subset  
    3. 90,016 spoof samples from SpoofCeleb train subset
    4. Keep other subsets (dev, test) unchanged
    
    Args:
        voxceleb2_dataset (pd.DataFrame): The voxceleb2 dataset to sample bonafide from
        spoof_celeb_dataset (pd.DataFrame): The spoof_celeb dataset to sample from
        random_state (int): Random seed for reproducibility
    
    Returns:
        pd.DataFrame: Complete dataset with balanced train subset and unchanged dev/test subsets
    """
    import pandas as pd
    import numpy as np
    
    # Set random seed for reproducibility
    np.random.seed(random_state)
    
    # Define target counts
    target_bonafide_spoofceleb = 45008
    target_bonafide_voxceleb2 = 45008
    target_spoof_spoofceleb = 90016
    
    print(f"🎯 SAMPLING TARGETS:")
    print(f"   • 45,008 bonafide samples from SpoofCeleb train subset")
    print(f"   • 45,008 bonafide samples from VoxCeleb2 train subset")
    print(f"   • 90,016 spoof samples from SpoofCeleb train subset")
    print(f"   • Total train samples: {target_bonafide_spoofceleb + target_bonafide_voxceleb2 + target_spoof_spoofceleb:,}")
    
    # Get VoxCeleb2 train samples (all bonafide)
    voxceleb2_train = voxceleb2_dataset[voxceleb2_dataset['subset'] == 'train'].copy()
    print(f"📊 VoxCeleb2 train samples available: {len(voxceleb2_train):,}")
    
    # Get SpoofCeleb train samples
    spoof_celeb_train = spoof_celeb_dataset[spoof_celeb_dataset['subset'] == 'train'].copy()
    
    # Split SpoofCeleb by labels
    bonafide_spoofceleb = spoof_celeb_train[spoof_celeb_train['label'] == 'bonafide'].copy()
    spoof_spoofceleb = spoof_celeb_train[spoof_celeb_train['label'] == 'spoof'].copy()
    
    print(f"📊 SpoofCeleb train samples available:")
    print(f"   • Bonafide samples: {len(bonafide_spoofceleb):,}")
    print(f"   • Spoof samples: {len(spoof_spoofceleb):,}")
    
    # Check if we have enough samples and adjust if necessary
    actual_bonafide_spoofceleb = min(target_bonafide_spoofceleb, len(bonafide_spoofceleb))
    actual_bonafide_voxceleb2 = min(target_bonafide_voxceleb2, len(voxceleb2_train))
    actual_spoof_spoofceleb = min(target_spoof_spoofceleb, len(spoof_spoofceleb))
    
    if actual_bonafide_spoofceleb < target_bonafide_spoofceleb:
        print(f"⚠️  Warning: Not enough bonafide samples in SpoofCeleb! Available: {len(bonafide_spoofceleb):,}, Needed: {target_bonafide_spoofceleb:,}")
    
    if actual_bonafide_voxceleb2 < target_bonafide_voxceleb2:
        print(f"⚠️  Warning: Not enough bonafide samples in VoxCeleb2! Available: {len(voxceleb2_train):,}, Needed: {target_bonafide_voxceleb2:,}")
    
    if actual_spoof_spoofceleb < target_spoof_spoofceleb:
        print(f"⚠️  Warning: Not enough spoof samples in SpoofCeleb! Available: {len(spoof_spoofceleb):,}, Needed: {target_spoof_spoofceleb:,}")
    
    print(f"📝 ACTUAL SAMPLING:")
    print(f"   • {actual_bonafide_spoofceleb:,} bonafide samples from SpoofCeleb train subset")
    print(f"   • {actual_bonafide_voxceleb2:,} bonafide samples from VoxCeleb2 train subset")
    print(f"   • {actual_spoof_spoofceleb:,} spoof samples from SpoofCeleb train subset")
    
    # Randomly sample the required number of samples
    print(f"🎲 Randomly sampling samples...")
    
    # Sample bonafide from SpoofCeleb
    sampled_bonafide_spoofceleb = bonafide_spoofceleb.sample(n=actual_bonafide_spoofceleb, random_state=random_state)
    
    # Sample bonafide from VoxCeleb2
    sampled_bonafide_voxceleb2 = voxceleb2_train.sample(n=actual_bonafide_voxceleb2, random_state=random_state)
    
    # Sample spoof from SpoofCeleb
    sampled_spoof_spoofceleb = spoof_spoofceleb.sample(n=actual_spoof_spoofceleb, random_state=random_state)
    
    # Combine all sampled train data
    balanced_train = pd.concat([
        sampled_bonafide_spoofceleb,
        sampled_bonafide_voxceleb2,
        sampled_spoof_spoofceleb
    ], ignore_index=True)
    
    # Shuffle the train subset
    balanced_train = balanced_train.sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    # Get other subsets (dev, eval) unchanged
    other_subsets = spoof_celeb_dataset[spoof_celeb_dataset['subset'] != 'train'].copy()
    
    # Combine with other subsets from VoxCeleb2
    voxceleb2_dev = voxceleb2_dataset[voxceleb2_dataset['subset'] == 'dev'].copy()
    voxceleb2_eval = voxceleb2_dataset[voxceleb2_dataset['subset'] == 'eval'].copy()
    # Combine balanced train with unchanged other subsets
    final_dataset = pd.concat([balanced_train, other_subsets, voxceleb2_dev, voxceleb2_eval], ignore_index=True)
    
    print(f"✅ Created balanced dataset:")
    print(f"   • Train subset: {len(balanced_train):,} samples")
    print(f"     - Bonafide from SpoofCeleb: {actual_bonafide_spoofceleb:,}")
    print(f"     - Bonafide from VoxCeleb2: {actual_bonafide_voxceleb2:,}")
    print(f"     - Spoof from SpoofCeleb: {actual_spoof_spoofceleb:,}")
    print(f"   • Other subsets: {len(other_subsets):,} samples (unchanged)")
    print(f"   • Total dataset: {len(final_dataset):,} samples")
    
    return final_dataset

# Create the balanced dataset
print("🚀 Creating balanced dataset with balanced train subset...")
balanced_spoof_celeb_dataset = create_balanced_train_dataset(voxceleb2_noise_dataset, spoof_celeb_dataset)

# Display the distribution of the balanced dataset
display_beautiful_table(balanced_spoof_celeb_dataset, "Balanced SpoofCeleb Dataset")


🚀 Creating balanced dataset with balanced train subset...
🎯 SAMPLING TARGETS:
   • 45,008 bonafide samples from SpoofCeleb train subset
   • 45,008 bonafide samples from VoxCeleb2 train subset
   • 90,016 spoof samples from SpoofCeleb train subset
   • Total train samples: 180,032
📊 VoxCeleb2 train samples available: 45,008
📊 SpoofCeleb train samples available:
   • Bonafide samples: 230,948
   • Spoof samples: 2,309,473
📝 ACTUAL SAMPLING:
   • 45,008 bonafide samples from SpoofCeleb train subset
   • 45,008 bonafide samples from VoxCeleb2 train subset
   • 90,016 spoof samples from SpoofCeleb train subset
🎲 Randomly sampling samples...
✅ Created balanced dataset:
   • Train subset: 180,032 samples
     - Bonafide from SpoofCeleb: 45,008
     - Bonafide from VoxCeleb2: 45,008
     - Spoof from SpoofCeleb: 90,016
   • Other subsets: 146,871 samples (unchanged)
   • Total dataset: 378,131 samples

📊 BALANCED SPOOFCELEB DATASET DATA DISTRIBUTION


,Dataset,Subset,Total,spoof_count,spoof_%,bonafide_count,bonafide_%
0,Balanced SpoofCeleb Dataset,train,180032,90016,50.0%,"90,016",50.0%
1,Balanced SpoofCeleb Dataset,dev,70732,47778,67.5%,"22,954",32.5%
2,Balanced SpoofCeleb Dataset,eval,127367,82017,64.4%,"45,350",35.6%
3,Balanced SpoofCeleb Dataset,TOTAL,378131,219811,58.1%,"158,320",41.9%



📈 SUMMARY STATISTICS:
   • Total samples: 378,131
   • Number of subsets: 3
   • Number of labels: 2
   • Labels: spoof, bonafide

🏷️  LABEL DISTRIBUTION:
   • spoof: 219,811 (58.1%)
   • bonafide: 158,320 (41.9%)


In [28]:
# Save the balanced dataset to a CSV file
output_path = '/nvme2/hungdx/Lightning-hydra/data/Wild_Spoof_Dataset/protocol.txt'
balanced_spoof_celeb_dataset.to_csv(output_path, index=False, sep=' ', header=False)
print(f"💾 Balanced SpoofCeleb dataset saved to: {output_path}")

# Show some sample data from the balanced dataset
print("\n📋 Sample data from balanced dataset:")
print(balanced_spoof_celeb_dataset.head(10))

# Additional statistics
print(f"\n📊 FINAL BALANCED DATASET STATISTICS:")
print(f"   • Total samples: {len(balanced_spoof_celeb_dataset):,}")

# Show train subset statistics
train_subset = balanced_spoof_celeb_dataset[balanced_spoof_celeb_dataset['subset'] == 'train']
print(f"\n🚂 TRAIN SUBSET STATISTICS:")
print(f"   • Train samples: {len(train_subset):,}")
print(f"   • Bonafide samples: {len(train_subset[train_subset['label'] == 'bonafide']):,}")
print(f"   • Spoof samples: {len(train_subset[train_subset['label'] == 'spoof']):,}")

# Show breakdown of bonafide sources
print(f"\n📋 TRAIN SUBSET BREAKDOWN:")
print(f"   • Bonafide samples: {len(train_subset[train_subset['label'] == 'bonafide']):,}")
print(f"     - From SpoofCeleb: 45,008")
print(f"     - From VoxCeleb2: 45,008")
print(f"   • Spoof samples: {len(train_subset[train_subset['label'] == 'spoof']):,}")
print(f"     - From SpoofCeleb: 90,016")

# Show other subsets statistics
other_subsets = balanced_spoof_celeb_dataset[balanced_spoof_celeb_dataset['subset'] != 'train']
print(f"\n📊 OTHER SUBSETS STATISTICS (unchanged):")
for subset in other_subsets['subset'].unique():
    subset_data = other_subsets[other_subsets['subset'] == subset]
    print(f"   • {subset}: {len(subset_data):,} samples")
    for label in subset_data['label'].unique():
        count = len(subset_data[subset_data['label'] == label])
        print(f"     - {label}: {count:,}")

# Compare with original datasets
print(f"\n🔄 COMPARISON WITH ORIGINAL DATASETS:")
print(f"   • VoxCeleb2 train samples available: {len(voxceleb2_noise_dataset[voxceleb2_noise_dataset['subset'] == 'train']):,}")
print(f"   • SpoofCeleb train bonafide available: {len(spoof_celeb_dataset[(spoof_celeb_dataset['subset'] == 'train') & (spoof_celeb_dataset['label'] == 'bonafide')]):,}")
print(f"   • SpoofCeleb train spoof available: {len(spoof_celeb_dataset[(spoof_celeb_dataset['subset'] == 'train') & (spoof_celeb_dataset['label'] == 'spoof')]):,}")
print(f"   • Final train samples: {len(train_subset):,}")
print(f"   • Expected: 180,032 (45,008 + 45,008 + 90,016)")
print(f"   • Match achieved: {'✅' if len(train_subset) == 180032 else '❌'}")


💾 Balanced SpoofCeleb dataset saved to: /nvme2/hungdx/Lightning-hydra/data/Wild_Spoof_Dataset/protocol.txt

📋 Sample data from balanced dataset:
                                           file_name subset     label
0  spoofceleb/flac/train/a04/id11111/G2QZRjUB_VM-...  train     spoof
1  spoofceleb/flac/train/a00/id10352/47enE06uMLU-...  train  bonafide
2  spoofceleb/flac/train/a05/id11226/HDEg5eXfSoc-...  train     spoof
3  spoofceleb/flac/train/a04/id10144/hMziHjQzJMs-...  train     spoof
4  spoofceleb/flac/train/a02/id10711/awK0SDaltDw-...  train     spoof
5  spoofceleb/flac/train/a00/id11223/yY2tTpDIs5c-...  train  bonafide
6  spoofceleb/flac/train/a00/id10213/BRUQ8X5w1fs-...  train  bonafide
7  spoofceleb/flac/train/a10/id10371/G5AlHTPuquM-...  train     spoof
8  spoofceleb/flac/train/a00/id10531/Awpe0P1eBfs-...  train  bonafide
9  spoofceleb/flac/train/a00/id10203/fvJBTVWPyD0-...  train  bonafide

📊 FINAL BALANCED DATASET STATISTICS:
   • Total samples: 378,131

🚂 TRAIN SUBSET STA

# Spoofceleb noisy

In [1]:
import pandas as pd

spoof_celeb_dataset = pd.read_csv('/home/hungdx/code/Lightning-hydra/data/Wild_Spoof_Dataset/spoofceleb_noise_dataset/protocol.txt', sep=' ', header=None)
spoof_celeb_dataset.columns = ['file_name', 'subset', 'label']


# Make a table to display the number of samples in each subset & label 

def create_data_distribution_table(dataset, dataset_name="Dataset"):
    """
    Create a beautiful table showing the distribution of samples across subsets and labels.
    
    Args:
        dataset (pd.DataFrame): The dataset to analyze
        dataset_name (str): Name of the dataset for display purposes
    
    Returns:
        pd.DataFrame: A formatted table with sample counts
    """
    import pandas as pd
    import numpy as np
    
    # Create cross-tabulation of subset and label
    crosstab = pd.crosstab(dataset['subset'], dataset['label'], margins=True)
    
    # Create a summary table
    summary_data = []
    
    # Add rows for each subset
    for subset in dataset['subset'].unique():
        subset_data = dataset[dataset['subset'] == subset]
        label_counts = subset_data['label'].value_counts()
        
        row = {
            'Dataset': dataset_name,
            'Subset': subset,
            'Total': len(subset_data)
        }
        
        # Add counts for each label
        for label in dataset['label'].unique():
            row[f'{label}_count'] = label_counts.get(label, 0)
            row[f'{label}_%'] = f"{(label_counts.get(label, 0) / len(subset_data) * 100):.1f}%"
        
        summary_data.append(row)
    
    # Add total row
    total_row = {
        'Dataset': dataset_name,
        'Subset': 'TOTAL',
        'Total': len(dataset)
    }
    
    for label in dataset['label'].unique():
        label_count = (dataset['label'] == label).sum()
        total_row[f'{label}_count'] = label_count
        total_row[f'{label}_%'] = f"{(label_count / len(dataset) * 100):.1f}%"
    
    summary_data.append(total_row)
    
    return pd.DataFrame(summary_data)

def display_beautiful_table(dataset, dataset_name="Dataset", show_percentages=True):
    """
    Display a beautiful, formatted table with the data distribution.
    
    Args:
        dataset (pd.DataFrame): The dataset to analyze
        dataset_name (str): Name of the dataset for display purposes
        show_percentages (bool): Whether to show percentage columns
    """
    import pandas as pd
    from IPython.display import display, HTML
    
    # Get the distribution table
    dist_table = create_data_distribution_table(dataset, dataset_name)
    
    # Style the table
    styled_table = dist_table.style.set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#4472C4'), ('color', 'black'), 
                                    ('font-weight', 'bold'), ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'center')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', 'black')]},
        {'selector': 'tr:nth-child(odd)', 'props': [('background-color', 'black')]},
        {'selector': 'tr:last-child', 'props': [('background-color', 'black'), 
                                               ('font-weight', 'bold')]}
    ])
    
    # Format numbers with thousands separators
    numeric_cols = [col for col in dist_table.columns if col.endswith('_count') or col == 'Total']
    for col in numeric_cols:
        styled_table = styled_table.format({col: '{:,}'})
    
    # Display the styled table
    print(f"\n📊 {dataset_name.upper()} DATA DISTRIBUTION")
    print("=" * 60)
    display(styled_table)
    
    # Print summary statistics
    print(f"\n📈 SUMMARY STATISTICS:")
    print(f"   • Total samples: {len(dataset):,}")
    print(f"   • Number of subsets: {dataset['subset'].nunique()}")
    print(f"   • Number of labels: {dataset['label'].nunique()}")
    print(f"   • Labels: {', '.join(dataset['label'].unique())}")
    
    # Show label distribution
    print(f"\n🏷️  LABEL DISTRIBUTION:")
    label_dist = dataset['label'].value_counts()
    for label, count in label_dist.items():
        percentage = (count / len(dataset)) * 100
        print(f"   • {label}: {count:,} ({percentage:.1f}%)")

# Display the distribution for spoof_celeb_dataset
display_beautiful_table(spoof_celeb_dataset, "SpoofCeleb Dataset")



📊 SPOOFCELEB DATASET DATA DISTRIBUTION


,Dataset,Subset,Total,spoof_count,spoof_%,bonafide_count,bonafide_%,eval_count,eval_%
0,SpoofCeleb Dataset,train,25403,23112,91.0%,2291,9.0%,0,0.0%
1,SpoofCeleb Dataset,dev,55740,47778,85.7%,7962,14.3%,0,0.0%
2,SpoofCeleb Dataset,eval,172274,0,0.0%,0,0.0%,"172,274",100.0%
3,SpoofCeleb Dataset,TOTAL,253417,70890,28.0%,10253,4.0%,"172,274",68.0%



📈 SUMMARY STATISTICS:
   • Total samples: 253,417
   • Number of subsets: 3
   • Number of labels: 3
   • Labels: spoof, bonafide, eval

🏷️  LABEL DISTRIBUTION:
   • eval: 172,274 (68.0%)
   • spoof: 70,890 (28.0%)
   • bonafide: 10,253 (4.0%)
